# PaperMind — Notebook 5: Multi-Document Agent

Notebook 2's router had **N static tools** stuffed directly into the LLM prompt: every query, every description. That works for 2–10 tools — but a real PaperMind library could have hundreds of papers, each with multiple tools (vector search, summary, etc.). Cramming hundreds of descriptions into every prompt is wasteful and starts to crowd out the actual question.

**A multi-document agent uses an `ObjectIndex` to retrieve tools dynamically.** Instead of all tools being visible at once, an embedding index over tool descriptions returns just the top-K most relevant tools per query. The agent then ReAct-chains over that small focused set.

**What we build:**
1. Three papers — *Attention Is All You Need*, *BERT*, and *RAG* (Lewis et al., 2020).
2. Two tools per paper:
   - **vector tool** — semantic top-k retrieval, for specific factual questions.
   - **summary tool** — `SummaryIndex` with tree summarisation, for high-level questions like *"what is this paper about?"*.
3. An `ObjectIndex` over all 6 tools — embeds tool *descriptions* and retrieves the most relevant ones per query.
4. A `ReActAgent` whose tool list is *dynamically* populated by the ObjectIndex retriever.
5. Three queries — single-paper, two-paper comparison, and a three-paper synthesis — each with a printout of which tools the agent ended up invoking.

## 1. Setup — env, LLM, embeddings

We use **Groq** (`llama-3.3-70b-versatile`) as the LLM provider. Cerebras's free tier hit unpredictable queue contention even for small workloads; Groq's free tier has a steadier 12 k TPM bucket which now fits comfortably thanks to the cheaper summary tools (see step 4).

Make sure `GROQ_API_KEY` is in `../.env`. No `nest_asyncio.apply()` — it conflicts with Groq's `httpx + anyio + sniffio` stack.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv("../.env")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
assert GROQ_API_KEY, "GROQ_API_KEY not found in ../.env"

from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

# Single LLM for everything — ReAct reasoning, vector tools, summary tools.
# After Option A (summary tools ride on vector index, not whole-paper SummaryIndex),
# each summary call uses ~3–4 k tokens. That fits inside Groq's 12 k TPM bucket
# even when the agent fires 2–3 calls per query, so a single provider is enough.
llm = Groq(model="llama-3.3-70b-versatile", api_key=GROQ_API_KEY)
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-base-en-v1.5")
Settings.llm = llm
Settings.embed_model = embed_model

print("Groq llama-3.3-70b-versatile + BGE embeddings configured")

Groq llama-3.3-70b-versatile + BGE embeddings configured


## 2. Papers — define metadata and ensure all three are downloaded

Two papers (`attention`, `bert`) were already pulled in notebook 2. We add the **RAG** paper (Lewis et al., 2020) — *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*.

Each paper has three on-disk locations: the PDF, a vector-index dir, and a summary-index dir.

In [2]:
import requests

DATA_DIR = Path("../data")
INDEX_ROOT = Path("../indexes")
DATA_DIR.mkdir(parents=True, exist_ok=True)
INDEX_ROOT.mkdir(parents=True, exist_ok=True)

papers = {
    "attention": {
        "title": "Attention Is All You Need",
        "authors": "Vaswani et al., 2017",
        "url": "https://arxiv.org/pdf/1706.03762",
    },
    "bert": {
        "title": "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding",
        "authors": "Devlin et al., 2018",
        "url": "https://arxiv.org/pdf/1810.04805",
    },
    "rag": {
        "title": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
        "authors": "Lewis et al., 2020",
        "url": "https://arxiv.org/pdf/2005.11401",
    },
}

for name, p in papers.items():
    p["pdf"] = DATA_DIR / f"{name}.pdf"
    p["vector_dir"] = INDEX_ROOT / name
    p["summary_dir"] = INDEX_ROOT / f"{name}_summary"

    if not p["pdf"].exists():
        r = requests.get(p["url"], timeout=60)
        r.raise_for_status()
        p["pdf"].write_bytes(r.content)
        print(f"[{name}] downloaded ({p['pdf'].stat().st_size / 1024:.1f} KB)")
    else:
        print(f"[{name}] already on disk")

[attention] already on disk
[bert] already on disk
[rag] already on disk


## 3. Per-paper indexes — vector AND summary

Two index types per paper, each persisted so re-runs are instant:

- **`VectorStoreIndex`** — embedding-based top-k retrieval. Best for narrow, factual queries.
- **`SummaryIndex`** — keeps the full document and runs tree-summarisation at query time. Best for high-level overviews. (Doesn't use embeddings; the indexing step is essentially free.)

Both indexes share the same source documents (PyMuPDF page extraction).

In [3]:
import fitz  # pymupdf
from llama_index.core import (
    Document,
    VectorStoreIndex,
    SummaryIndex,
    StorageContext,
    load_index_from_storage,
)


def load_pdf_as_documents(pdf_path: Path) -> list[Document]:
    pdf = fitz.open(str(pdf_path))
    docs = [
        Document(text=page.get_text(), metadata={"page": i + 1, "source": pdf_path.name})
        for i, page in enumerate(pdf)
    ]
    pdf.close()
    return docs


def build_or_load(index_cls, persist_dir: Path, docs_factory):
    persist_dir.mkdir(parents=True, exist_ok=True)
    if any(persist_dir.iterdir()):
        ctx = StorageContext.from_defaults(persist_dir=str(persist_dir))
        return load_index_from_storage(ctx)
    docs = docs_factory()
    idx = index_cls.from_documents(docs)
    idx.storage_context.persist(persist_dir=str(persist_dir))
    return idx


for name, p in papers.items():
    print(f"[{name}]")
    # cache docs so we read the PDF at most once per paper
    cached = {"docs": None}
    def _docs(pdf=p["pdf"], cache=cached):
        if cache["docs"] is None:
            cache["docs"] = load_pdf_as_documents(pdf)
        return cache["docs"]

    p["vector_index"] = build_or_load(VectorStoreIndex, p["vector_dir"], _docs)
    print(f"  vector index ready  ({p['vector_dir']})")
    p["summary_index"] = build_or_load(SummaryIndex, p["summary_dir"], _docs)
    print(f"  summary index ready ({p['summary_dir']})")

[attention]
  vector index ready  (../indexes/attention)
  summary index ready (../indexes/attention_summary)
[bert]
  vector index ready  (../indexes/bert)
  summary index ready (../indexes/bert_summary)
[rag]
  vector index ready  (../indexes/rag)
  summary index ready (../indexes/rag_summary)


## 4. Build six tools — vector + summary per paper

Two tools per paper, with descriptions written *to be embedded and retrieved*. The descriptions are the only thing the `ObjectIndex` sees, so they should:

- name the paper (title, authors, year, key concepts)
- state the *kind* of question the tool is good for (specific facts vs holistic summaries)
- avoid generic boilerplate that would inflate similarity to unrelated queries

**Vector tools also get a reranker.** BGE-base is fast but sometimes pulls chunks that *look* topically related (e.g. anything mentioning "decoding" or "RAG") instead of the chunk that actually answers the question. A cross-encoder reranker takes the query + each candidate chunk together and scores them more precisely. We retrieve top-10 with BGE, then rerank to top-3.

The reranker model (`cross-encoder/ms-marco-MiniLM-L-6-v2`, ~90 MB) downloads on first use and is cached for subsequent runs.

In [4]:
from llama_index.core.tools import QueryEngineTool
from llama_index.core.postprocessor import SentenceTransformerRerank

# One reranker shared across all vector tools.
# Retrieve top-10 with BGE, then keep the 3 most genuinely relevant per query.
reranker = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_n=3,
)

TOOL_HINTS = {
    "attention": {
        "vector": (
            "Specific factual questions about 'Attention Is All You Need' (Vaswani et al., 2017): "
            "scaled dot-product attention math, multi-head attention head counts, sinusoidal positional encoding, "
            "encoder-decoder layer counts, training hyperparameters, BLEU scores on WMT 2014 EN-DE / EN-FR."
        ),
        "summary": (
            "High-level summary of 'Attention Is All You Need' (Vaswani et al., 2017). Use for questions like "
            "'what is this paper about?', 'what are the main contributions?', or 'summarise the approach'."
        ),
    },
    "bert": {
        "vector": (
            "Specific factual questions about BERT (Devlin et al., 2018): masked-language-modeling masking ratios, "
            "next-sentence-prediction details, BERT-Base vs BERT-Large parameter counts, GLUE / SQuAD scores, "
            "WordPiece tokenization, fine-tuning hyperparameters."
        ),
        "summary": (
            "High-level summary of the BERT paper (Devlin et al., 2018). Use for 'what is BERT?', 'what is the "
            "core idea?', or 'summarise the contributions of this paper'."
        ),
    },
    "rag": {
        "vector": (
            "Specific factual questions about 'Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks' "
            "(Lewis et al., 2020): RAG-Sequence vs RAG-Token decoding, the DPR retriever (BERT query/doc encoders), "
            "BART generator, marginalisation over retrieved documents, performance on Natural Questions / "
            "TriviaQA / Jeopardy / FEVER."
        ),
        "summary": (
            "High-level summary of the RAG paper (Lewis et al., 2020). Use for 'what is RAG?', 'how does RAG "
            "combine retrieval with generation?', or 'summarise the contributions'."
        ),
    },
}

all_tools: list[QueryEngineTool] = []
for name, p in papers.items():
    # Vector tool: precise factual retrieval — top-10 → rerank to top 3
    vector_tool = QueryEngineTool.from_defaults(
        query_engine=p["vector_index"].as_query_engine(
            similarity_top_k=10,
            node_postprocessors=[reranker],
        ),
        name=f"{name}_vector",
        description=TOOL_HINTS[name]["vector"],
    )
    # Summary tool: also rides on the vector index, but pulls a wider top-10 and
    # uses tree_summarize over those chunks. ~3–4 k tokens per call instead of
    # ~10 k for full-paper SummaryIndex — much friendlier to free-tier rate
    # limits, and for these queries the answer quality is essentially the same.
    summary_tool = QueryEngineTool.from_defaults(
        query_engine=p["vector_index"].as_query_engine(
            similarity_top_k=10,
            response_mode="tree_summarize",
        ),
        name=f"{name}_summary",
        description=TOOL_HINTS[name]["summary"],
    )
    all_tools.extend([vector_tool, summary_tool])

print(f"Built {len(all_tools)} tools (vector→rerank, summary→top-10+tree_summarize):")
for t in all_tools:
    print(f"  • {t.metadata.name}")

Built 6 tools (vector→rerank, summary→top-10+tree_summarize):
  • attention_vector
  • attention_summary
  • bert_vector
  • bert_summary
  • rag_vector
  • rag_summary


## 5. ObjectIndex — embed tool descriptions for dynamic retrieval

`ObjectIndex.from_objects(...)` builds an embedding index *over the tool metadata*. At query time, `as_retriever(similarity_top_k=K)` returns the K tools whose descriptions are most similar to the user's question.

This is the same BGE embedding model we use elsewhere — the tool selection is just another vector-similarity problem.

In [5]:
from llama_index.core.objects import ObjectIndex

obj_index = ObjectIndex.from_objects(
    objects=all_tools,
    index_cls=VectorStoreIndex,
)
tool_retriever = obj_index.as_retriever(similarity_top_k=3)

# quick smoke test — what does the retriever return for a sample query?
preview_query = "How does masked language modelling work in BERT?"
retrieved = tool_retriever.retrieve(preview_query)
print(f"Tools retrieved for: {preview_query!r}")
for t in retrieved:
    print(f"  • {t.metadata.name}")

Tools retrieved for: 'How does masked language modelling work in BERT?'
  • bert_vector
  • bert_summary
  • rag_vector


## 6. Build the agent with `tool_retriever`

Same `ReActAgent` from notebook 4, but instead of a static `tools=[...]` list we pass `tool_retriever`. Per query, the agent:

1. Calls `tool_retriever.retrieve(query)` to get the top-K relevant tools.
2. Runs its ReAct loop using only those tools.

This means each prompt only contains a handful of tool descriptions — even if the underlying tool pool has hundreds.

In [6]:
from llama_index.core.agent.workflow import (
    ReActAgent,
    AgentStream,
    ToolCall,
    ToolCallResult,
)

agent = ReActAgent(tool_retriever=tool_retriever, llm=llm)
print("Agent built with dynamic tool retrieval over", len(all_tools), "tools")

Agent built with dynamic tool retrieval over 6 tools


## 7. Run queries — track which tools the agent actually used

Define the trace helper once, then run each query in its own cell. Splitting them lets you re-run individual queries when Cerebras's free-tier queue is busy without re-doing the ones that already succeeded.

The helper streams events from `agent.run(...)` and dispatches:
- `ToolCall` → records which tool the agent picked + its arguments
- `ToolCallResult` → the Observation (tool output, preview-truncated)

After events are exhausted, `await handler` resolves to the final `AgentOutput`.

**Three queries by design:**
1. **Single-paper deep question** — should hit just RAG.
2. **Two-paper comparison** — should hit BERT + Transformer.
3. **Three-paper synthesis** — should pull tools from all three papers.

In [7]:
async def run_with_tool_trace(query: str):
    print("=" * 100)
    print(f"Q: {query}\n")

    pre_retrieved = [t.metadata.name for t in tool_retriever.retrieve(query)]
    print(f"Pre-retrieved tool candidates: {pre_retrieved}\n")

    tool_calls: list[tuple[str, dict]] = []
    handler = agent.run(user_msg=query)

    async for ev in handler.stream_events():
        if isinstance(ev, ToolCall):
            tool_calls.append((ev.tool_name, dict(ev.tool_kwargs)))
            print(f"  → {ev.tool_name}({ev.tool_kwargs})")
        elif isinstance(ev, ToolCallResult):
            obs = str(ev.tool_output).replace("\n", " ")
            preview = obs if len(obs) <= 240 else obs[:240] + " ..."
            print(f"    obs: {preview}\n")

    response = await handler
    print(f"--- {len(tool_calls)} tool calls used ---")
    for name, _ in tool_calls:
        print(f"  • {name}")
    print(f"\n--- Final answer ---\n{response}\n")
    return response

In [8]:
# Query 1 — single-paper deep question. Should hit just rag_vector.
await run_with_tool_trace(
    "In the RAG paper, how does RAG-Sequence differ from RAG-Token at decoding time?"
)

Q: In the RAG paper, how does RAG-Sequence differ from RAG-Token at decoding time?

Pre-retrieved tool candidates: ['rag_vector', 'rag_summary', 'bert_vector']

  → rag_vector({'input': 'RAG-Sequence vs RAG-Token decoding'})
    obs: h x T h z where h x and h z are the vector representations of the input x and the document z, respectively. The retriever is trained using a contrastive loss function. 2.3 Generator: Transformer The generator pθ(y|x, z) is based on a standa ...

--- 1 tool calls used ---
  • rag_vector

--- Final answer ---
The RAG-Sequence model uses the same retrieved document to generate the complete sequence, whereas the RAG-Token model can draw a different latent document for each target token and marginalize accordingly, allowing it to choose content from several documents when producing an answer.



AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='The RAG-Sequence model uses the same retrieved document to generate the complete sequence, whereas the RAG-Token model can draw a different latent document for each target token and marginalize accordingly, allowing it to choose content from several documents when producing an answer.')]), structured_response=None, current_agent_name='Agent', raw={'id': 'chatcmpl-7123d09a-e3ca-4210-b401-843702118f6c', 'choices': [{'delta': {'content': None, 'function_call': None, 'refusal': None, 'role': None, 'tool_calls': None}, 'finish_reason': 'stop', 'index': 0, 'logprobs': None}], 'created': 1778528929, 'model': 'llama-3.3-70b-versatile', 'object': 'chat.completion.chunk', 'service_tier': None, 'system_fingerprint': 'fp_ba38bbab80', 'usage': {'completion_tokens': 93, 'prompt_tokens': 1226, 'total_tokens': 1319, 'completion_tokens_details': None, 'prompt_toke

In [9]:
# Query 2 — two-paper comparison. Should hit BERT + Transformer tools.
await run_with_tool_trace(
    "How does BERT's bidirectional pre-training objective differ from the "
    "original Transformer's encoder-decoder training, and why does that matter?"
)

Q: How does BERT's bidirectional pre-training objective differ from the original Transformer's encoder-decoder training, and why does that matter?

Pre-retrieved tool candidates: ['bert_vector', 'rag_vector', 'bert_summary']

  → bert_vector({'input': 'BERT pre-training objective vs original Transformer encoder-decoder training'})
    obs: The BERT pre-training objective is different from the original Transformer encoder-decoder training. In the original Transformer, the encoder-decoder training is typically done using a left-to-right language model, where the model is traine ...

--- 1 tool calls used ---
  • bert_vector

--- Final answer ---
BERT's bidirectional pre-training objective differs from the original Transformer's encoder-decoder training in that it uses a masked language model (MLM) to predict the original vocabulary id of a masked word based on its context, allowing the model to fuse left and right context. Additionally, BERT uses a next sentence prediction task to jointl

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="BERT's bidirectional pre-training objective differs from the original Transformer's encoder-decoder training in that it uses a masked language model (MLM) to predict the original vocabulary id of a masked word based on its context, allowing the model to fuse left and right context. Additionally, BERT uses a next sentence prediction task to jointly pre-train text-pair representations. This difference in pre-training objectives enables BERT to capture more contextual information and achieve state-of-the-art results on a wide range of natural language processing tasks.")]), structured_response=None, current_agent_name='Agent', raw={'id': 'chatcmpl-42e1e26b-c5f6-4cbb-aea6-8d04e913aa2a', 'choices': [{'delta': {'content': None, 'function_call': None, 'refusal': None, 'role': None, 'tool_calls': None}, 'finish_reason': 'stop', 'index': 0, 'logprobs': Non

In [10]:
# Query 3 — three-paper synthesis. Heaviest of the three: usually pulls tools
# from all three papers and may invoke summary tools, so the most likely to
# bump the Cerebras queue. Re-run this cell on its own if it 429s.
await run_with_tool_trace(
    "Trace the evolution from the original Transformer to BERT to RAG. "
    "What does each paper contribute toward retrieval-augmented language "
    "models, and how do they build on each other?"
)

Q: Trace the evolution from the original Transformer to BERT to RAG. What does each paper contribute toward retrieval-augmented language models, and how do they build on each other?

Pre-retrieved tool candidates: ['rag_vector', 'bert_vector', 'rag_summary']

  → rag_summary({'input': 'What is RAG and how does it combine retrieval with generation?'})
    obs: RAG, or Retrieval-Augmented Generation, is a model that combines a pre-trained retriever with a pre-trained sequence-to-sequence model, known as a generator. The retriever fetches relevant documents from a non-parametric memory, which is a  ...

  → bert_vector({'input': 'What are the key contributions of BERT to the development of retrieval-augmented language models?'})
    obs: The key contributions of BERT are demonstrating the importance of bidirectional pre-training for language representations, showing that pre-trained representations reduce the need for many heavily-engineered task-specific architectures, and ...

  → rag_v

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='The original Transformer introduced self-attention mechanisms, which are used in BERT to learn contextualized representations through bidirectional pre-training. BERT, in turn, provides a pre-trained language model that can be fine-tuned for specific tasks, including those that involve retrieval-augmented generation. RAG builds on BERT by combining pre-trained parametric and non-parametric memory, using a pre-trained seq2seq model and a dense vector index of knowledge, to enable retrieval-augmented generation. This allows RAG to leverage the strengths of both parametric and non-parametric approaches, enabling it to generate more specific, diverse, and factual language.')]), structured_response=None, current_agent_name='Agent', raw={'id': 'chatcmpl-08359a64-73fa-482b-b2df-50f98ee95482', 'choices': [{'delta': {'content': None, 'function_call': None,

## How this differs from the router (notebook 2)

| | **Router** (nb 2) | **Multi-document agent** (this nb) |
|---|---|---|
| Tool list | static, all in prompt | dynamic, top-K retrieved per query |
| Picks how many | exactly one (`LLMSingleSelector`) | as many as the ReAct loop needs |
| Tool variety per paper | one query engine per paper | vector + summary (or more) per paper |
| Scales to N tools | breaks down past ~10 (prompt bloat) | scales to 100s — descriptions are embedded |
| Tool selection mechanism | LLM reading all descriptions | embedding similarity → narrow → ReAct |
| Cost per query | 1 LLM call (router) + 1 (synthesis) | 1 retriever lookup + N ReAct LLM calls |

**When the router is enough:** small fixed set of sources; queries map cleanly to one source; latency matters.

**When you want a multi-document agent:**
- The library grows over time (new papers, new tools) and you don't want to reflow the system prompt.
- Multiple tools per source — vector for facts, summary for overviews, maybe later a citation extractor or table reader.
- Queries genuinely need *several* tools chained — synthesis questions like the third one above.

**Tuning levers** (in order of impact):
1. **Tool descriptions** — they're the entire signal the retriever sees. Same advice as everywhere else: be specific, name the paper, state the *kind* of query the tool is for.
2. **`similarity_top_k`** on the tool retriever — too low and the agent misses relevant tools; too high and you negate the prompt-bloat advantage. 3–5 is usually right.
3. **Embedding model** — the same `BAAI/bge-base-en-v1.5` we use for content retrieval. If tool selection is noisy, a stronger embedding model helps.